# ZGLMM381 — Evolucao de RFQ — cotacoes e respostas de fornecedor — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq`
**Transação SAP:** ZGLMM381
**Colunas:** 23 · **Clustering declarado:** **_(nenhum declarado)_**

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **Nenhum clustering declarado.** É a única das cinco sem chave de agrupamento definida — a seção 4 testa quatro candidatas para descobrir qual identifica a linha. |
| 🔴 ALTA | **Nenhuma coluna de auditoria de carga** — sem `dateingest`, `dh_carga` ou equivalente (seção 17). |
| 🟡 MEDIA | **22 das 23 colunas estão sem comentário** no `DESCRIBE`. Apenas `cod_centro` tem descrição. Sem dicionário, o de/para com o SAP depende de inferência pelo nome. |
| 🟡 MEDIA | **`vl_cotacao` é `decimal(38,6)`** — precisão muito acima do usual para valor monetário (o normal seria `decimal(18,2)`). Confirmar se há uso real dessas casas. |
| ⚪ BAIXA | **`qt_rodadas_fornecedor` é `bigint`** — sugere contagem de rodadas de negociação; vale confirmar o domínio real (seção 18.2). |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 12.2 | Campos de hora |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `num_solicitacao_cotacao_coletiva`, `num_cotacao`, `num_item_cotacao`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro`)                                 AS valores_de_cod_centro,
       COUNT(DISTINCT `num_solicitacao_cotacao_coletiva`)                                 AS num_solicitacao_cotacao_coletiva_distintos
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq),
g AS (
  SELECT 'num_cotacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_cotacao + num_item_cotacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_cotacao`, `num_item_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_cotacao + num_item_cotacao + cod_fornecedor' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_cotacao`, `num_item_cotacao`, `cod_fornecedor` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_solicitacao_cotacao_coletiva + num_cotacao + num_item_cotacao' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_solicitacao_cotacao_coletiva`, `num_cotacao`, `num_item_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_cotacao + num_item_cotacao`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_cotacao`, `num_item_cotacao`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY `num_cotacao`, `num_item_cotacao`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (
  SELECT `num_cotacao`, `num_item_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `num_cotacao`, `num_item_cotacao` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq c JOIN dup ON c.`num_cotacao` <=> dup.`num_cotacao` AND c.`num_item_cotacao` <=> dup.`num_item_cotacao`
),
agg AS (
  SELECT `num_cotacao`, `num_item_cotacao`,
         COUNT(DISTINCT `num_solicitacao_cotacao_coletiva`) AS `num_solicitacao_cotacao_coletiva`,
         COUNT(DISTINCT `num_requisicao_compra`) AS `num_requisicao_compra`,
         COUNT(DISTINCT `num_item_requisicao_compra`) AS `num_item_requisicao_compra`,
         COUNT(DISTINCT `vl_cotacao`) AS `vl_cotacao`,
         COUNT(DISTINCT `cod_condicao_pagamento`) AS `cod_condicao_pagamento`,
         COUNT(DISTINCT `dt_criacao_cotacao`) AS `dt_criacao_cotacao`,
         COUNT(DISTINCT `dt_prazo_apresentacao_cotacao`) AS `dt_prazo_apresentacao_cotacao`,
         COUNT(DISTINCT `hora_fechamento_cotacao`) AS `hora_fechamento_cotacao`,
         COUNT(DISTINCT `st_resposta_fornecedor`) AS `st_resposta_fornecedor`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `cod_usuario_criacao`) AS `cod_usuario_criacao`,
         COUNT(DISTINCT `st_cotacao`) AS `st_cotacao`,
         COUNT(DISTINCT `cod_material`) AS `cod_material`,
         COUNT(DISTINCT `desc_item`) AS `desc_item`,
         COUNT(DISTINCT `qt_rodadas_fornecedor`) AS `qt_rodadas_fornecedor`,
         COUNT(DISTINCT `nm_titulo`) AS `nm_titulo`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_grupo_fornecedores`) AS `cod_grupo_fornecedores`,
         COUNT(DISTINCT `dt_criacao_documento_subsequente`) AS `dt_criacao_documento_subsequente`
  FROM d GROUP BY `num_cotacao`, `num_item_cotacao`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(21,
    'num_solicitacao_cotacao_coletiva', MAX(`num_solicitacao_cotacao_coletiva`),
    'num_requisicao_compra', MAX(`num_requisicao_compra`),
    'num_item_requisicao_compra', MAX(`num_item_requisicao_compra`),
    'vl_cotacao', MAX(`vl_cotacao`),
    'cod_condicao_pagamento', MAX(`cod_condicao_pagamento`),
    'dt_criacao_cotacao', MAX(`dt_criacao_cotacao`),
    'dt_prazo_apresentacao_cotacao', MAX(`dt_prazo_apresentacao_cotacao`),
    'hora_fechamento_cotacao', MAX(`hora_fechamento_cotacao`),
    'st_resposta_fornecedor', MAX(`st_resposta_fornecedor`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'cod_usuario_criacao', MAX(`cod_usuario_criacao`),
    'st_cotacao', MAX(`st_cotacao`),
    'cod_material', MAX(`cod_material`),
    'desc_item', MAX(`desc_item`),
    'qt_rodadas_fornecedor', MAX(`qt_rodadas_fornecedor`),
    'nm_titulo', MAX(`nm_titulo`),
    'cod_empresa', MAX(`cod_empresa`),
    'cod_centro', MAX(`cod_centro`),
    'cod_grupo_fornecedores', MAX(`cod_grupo_fornecedores`),
    'dt_criacao_documento_subsequente', MAX(`dt_criacao_documento_subsequente`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

> **22 colunas não têm comentário** no `DESCRIBE` desta tabela. Coluna sem descrição costuma ser a assinatura de campo nunca carregado — confira o `veredito` delas com atenção.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq),
perf AS (
  SELECT stack(23,
    'num_solicitacao_cotacao_coletiva', 'string', COUNT_IF(`num_solicitacao_cotacao_coletiva` IS NULL), COUNT_IF(`num_solicitacao_cotacao_coletiva` IS NOT NULL AND lower(trim(`num_solicitacao_cotacao_coletiva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_solicitacao_cotacao_coletiva`) RLIKE '^0+([.,]0+)?$'),
    'num_cotacao', 'string', COUNT_IF(`num_cotacao` IS NULL), COUNT_IF(`num_cotacao` IS NOT NULL AND lower(trim(`num_cotacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_cotacao`) RLIKE '^0+([.,]0+)?$'),
    'num_item_cotacao', 'string', COUNT_IF(`num_item_cotacao` IS NULL), COUNT_IF(`num_item_cotacao` IS NOT NULL AND lower(trim(`num_item_cotacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_cotacao`) RLIKE '^0+([.,]0+)?$'),
    'num_requisicao_compra', 'string', COUNT_IF(`num_requisicao_compra` IS NULL), COUNT_IF(`num_requisicao_compra` IS NOT NULL AND lower(trim(`num_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_requisicao_compra', 'string', COUNT_IF(`num_item_requisicao_compra` IS NULL), COUNT_IF(`num_item_requisicao_compra` IS NOT NULL AND lower(trim(`num_item_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_requisicao_compra`) RLIKE '^0+([.,]0+)?$'),
    'vl_cotacao', 'decimal(38,6)', COUNT_IF(`vl_cotacao` IS NULL), 0L, COUNT_IF(`vl_cotacao` = 0),
    'cod_condicao_pagamento', 'string', COUNT_IF(`cod_condicao_pagamento` IS NULL), COUNT_IF(`cod_condicao_pagamento` IS NOT NULL AND lower(trim(`cod_condicao_pagamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_condicao_pagamento`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao_cotacao', 'date', COUNT_IF(`dt_criacao_cotacao` IS NULL), 0L, 0L,
    'dt_prazo_apresentacao_cotacao', 'date', COUNT_IF(`dt_prazo_apresentacao_cotacao` IS NULL), 0L, 0L,
    'hora_fechamento_cotacao', 'string', COUNT_IF(`hora_fechamento_cotacao` IS NULL), COUNT_IF(`hora_fechamento_cotacao` IS NOT NULL AND lower(trim(`hora_fechamento_cotacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`hora_fechamento_cotacao`) RLIKE '^0+([.,]0+)?$'),
    'st_resposta_fornecedor', 'string', COUNT_IF(`st_resposta_fornecedor` IS NULL), COUNT_IF(`st_resposta_fornecedor` IS NOT NULL AND lower(trim(`st_resposta_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_resposta_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_usuario_criacao', 'string', COUNT_IF(`cod_usuario_criacao` IS NULL), COUNT_IF(`cod_usuario_criacao` IS NOT NULL AND lower(trim(`cod_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'st_cotacao', 'string', COUNT_IF(`st_cotacao` IS NULL), COUNT_IF(`st_cotacao` IS NOT NULL AND lower(trim(`st_cotacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_cotacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_item', 'string', COUNT_IF(`desc_item` IS NULL), COUNT_IF(`desc_item` IS NOT NULL AND lower(trim(`desc_item`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_item`) RLIKE '^0+([.,]0+)?$'),
    'qt_rodadas_fornecedor', 'bigint', COUNT_IF(`qt_rodadas_fornecedor` IS NULL), 0L, COUNT_IF(`qt_rodadas_fornecedor` = 0),
    'nm_titulo', 'string', COUNT_IF(`nm_titulo` IS NULL), COUNT_IF(`nm_titulo` IS NOT NULL AND lower(trim(`nm_titulo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_titulo`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_fornecedores', 'string', COUNT_IF(`cod_grupo_fornecedores` IS NULL), COUNT_IF(`cod_grupo_fornecedores` IS NOT NULL AND lower(trim(`cod_grupo_fornecedores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_fornecedores`) RLIKE '^0+([.,]0+)?$'),
    'dt_criacao_documento_subsequente', 'date', COUNT_IF(`dt_criacao_documento_subsequente` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq),
card AS (
  SELECT stack(23,
    'num_solicitacao_cotacao_coletiva', 'string', approx_count_distinct(`num_solicitacao_cotacao_coletiva`),
    'num_cotacao', 'string', approx_count_distinct(`num_cotacao`),
    'num_item_cotacao', 'string', approx_count_distinct(`num_item_cotacao`),
    'num_requisicao_compra', 'string', approx_count_distinct(`num_requisicao_compra`),
    'num_item_requisicao_compra', 'string', approx_count_distinct(`num_item_requisicao_compra`),
    'vl_cotacao', 'decimal(38,6)', approx_count_distinct(`vl_cotacao`),
    'cod_condicao_pagamento', 'string', approx_count_distinct(`cod_condicao_pagamento`),
    'dt_criacao_cotacao', 'date', approx_count_distinct(`dt_criacao_cotacao`),
    'dt_prazo_apresentacao_cotacao', 'date', approx_count_distinct(`dt_prazo_apresentacao_cotacao`),
    'hora_fechamento_cotacao', 'string', approx_count_distinct(`hora_fechamento_cotacao`),
    'st_resposta_fornecedor', 'string', approx_count_distinct(`st_resposta_fornecedor`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'cod_usuario_criacao', 'string', approx_count_distinct(`cod_usuario_criacao`),
    'st_cotacao', 'string', approx_count_distinct(`st_cotacao`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_item', 'string', approx_count_distinct(`desc_item`),
    'qt_rodadas_fornecedor', 'bigint', approx_count_distinct(`qt_rodadas_fornecedor`),
    'nm_titulo', 'string', approx_count_distinct(`nm_titulo`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_fornecedores', 'string', approx_count_distinct(`cod_grupo_fornecedores`),
    'dt_criacao_documento_subsequente', 'date', approx_count_distinct(`dt_criacao_documento_subsequente`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

In [0]:
-- DOMINIO DAS CATEGORICAS (top 8 de cada)
(SELECT 'st_resposta_fornecedor' AS coluna, CAST(`st_resposta_fornecedor` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `st_resposta_fornecedor` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_cotacao' AS coluna, CAST(`st_cotacao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `st_cotacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_condicao_pagamento' AS coluna, CAST(`cod_condicao_pagamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `cod_condicao_pagamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_fornecedores' AS coluna, CAST(`cod_grupo_fornecedores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY `cod_grupo_fornecedores` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(2,
    'vl_cotacao', 'decimal(38,6)', COUNT(`vl_cotacao`), CAST(MIN(`vl_cotacao`) AS DOUBLE), CAST(MAX(`vl_cotacao`) AS DOUBLE), CAST(AVG(`vl_cotacao`) AS DOUBLE), CAST(percentile_approx(`vl_cotacao`, 0.5) AS DOUBLE), COUNT_IF(`vl_cotacao` < 0), COUNT_IF(`vl_cotacao` = 0),
    'qt_rodadas_fornecedor', 'bigint', COUNT(`qt_rodadas_fornecedor`), CAST(MIN(`qt_rodadas_fornecedor`) AS DOUBLE), CAST(MAX(`qt_rodadas_fornecedor`) AS DOUBLE), CAST(AVG(`qt_rodadas_fornecedor`) AS DOUBLE), CAST(percentile_approx(`qt_rodadas_fornecedor`, 0.5) AS DOUBLE), COUNT_IF(`qt_rodadas_fornecedor` < 0), COUNT_IF(`qt_rodadas_fornecedor` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(2,
    'vl_cotacao', CAST(SUM(`vl_cotacao`) AS DOUBLE), CAST(ROUND(SUM(`vl_cotacao`), 2) AS STRING), COUNT(`vl_cotacao`),
    'qt_rodadas_fornecedor', CAST(SUM(`qt_rodadas_fornecedor`) AS DOUBLE), CAST(ROUND(SUM(`qt_rodadas_fornecedor`), 2) AS STRING), COUNT(`qt_rodadas_fornecedor`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
ORDER BY coluna;

## 12. Datas

Todas as datas desta tabela estão em tipo nativo `date`. A coluna `sentinela_9999`
conta ocorrências de `9999-12-31`, usada pelo SAP para "sem prazo definido".

In [0]:
-- DATAS EM TIPO NATIVO
SELECT coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999
FROM (
  SELECT stack(3,
    'dt_criacao_cotacao', COUNT_IF(`dt_criacao_cotacao` IS NULL), CAST(MIN(`dt_criacao_cotacao`) AS STRING), CAST(MAX(`dt_criacao_cotacao`) AS STRING), COUNT(DISTINCT `dt_criacao_cotacao`), COUNT_IF(`dt_criacao_cotacao` > current_date()), COUNT_IF(`dt_criacao_cotacao` = DATE'9999-12-31'),
    'dt_prazo_apresentacao_cotacao', COUNT_IF(`dt_prazo_apresentacao_cotacao` IS NULL), CAST(MIN(`dt_prazo_apresentacao_cotacao`) AS STRING), CAST(MAX(`dt_prazo_apresentacao_cotacao`) AS STRING), COUNT(DISTINCT `dt_prazo_apresentacao_cotacao`), COUNT_IF(`dt_prazo_apresentacao_cotacao` > current_date()), COUNT_IF(`dt_prazo_apresentacao_cotacao` = DATE'9999-12-31'),
    'dt_criacao_documento_subsequente', COUNT_IF(`dt_criacao_documento_subsequente` IS NULL), CAST(MIN(`dt_criacao_documento_subsequente`) AS STRING), CAST(MAX(`dt_criacao_documento_subsequente`) AS STRING), COUNT(DISTINCT `dt_criacao_documento_subsequente`), COUNT_IF(`dt_criacao_documento_subsequente` > current_date()), COUNT_IF(`dt_criacao_documento_subsequente` = DATE'9999-12-31')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
ORDER BY coluna;

## 12.2 Campos de hora

In [0]:
-- CAMPOS DE HORA
SELECT coluna, vazios, fmt_HHMMSS, fmt_HH_MM, hora_zero, distintos
FROM (
  SELECT stack(1,
    'hora_fechamento_cotacao', COUNT_IF(`hora_fechamento_cotacao` IS NULL OR trim(`hora_fechamento_cotacao`) = ''), COUNT_IF(trim(`hora_fechamento_cotacao`) RLIKE '^[0-9]{6}$'), COUNT_IF(trim(`hora_fechamento_cotacao`) RLIKE '^[0-9]{2}:[0-9]{2}'), COUNT_IF(trim(`hora_fechamento_cotacao`) IN ('000000', '00:00:00')), COUNT(DISTINCT `hora_fechamento_cotacao`)
  ) AS (coluna, vazios, fmt_HHMMSS, fmt_HH_MM, hora_zero, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(9,
    'num_solicitacao_cotacao_coletiva', 'string', COUNT_IF(CAST(`num_solicitacao_cotacao_coletiva` AS STRING) IS NULL OR trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING)) = ''), MIN(length(trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING)))), MAX(length(trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING)))), COUNT_IF(trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_solicitacao_cotacao_coletiva` AS STRING)), '^0+', '')),
    'num_cotacao', 'string', COUNT_IF(CAST(`num_cotacao` AS STRING) IS NULL OR trim(CAST(`num_cotacao` AS STRING)) = ''), MIN(length(trim(CAST(`num_cotacao` AS STRING)))), MAX(length(trim(CAST(`num_cotacao` AS STRING)))), COUNT_IF(trim(CAST(`num_cotacao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_cotacao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_cotacao` AS STRING)), '^0+', '')),
    'num_item_cotacao', 'string', COUNT_IF(CAST(`num_item_cotacao` AS STRING) IS NULL OR trim(CAST(`num_item_cotacao` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_cotacao` AS STRING)))), MAX(length(trim(CAST(`num_item_cotacao` AS STRING)))), COUNT_IF(trim(CAST(`num_item_cotacao` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_cotacao` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_cotacao` AS STRING)), '^0+', '')),
    'num_requisicao_compra', 'string', COUNT_IF(CAST(`num_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`num_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`num_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_requisicao_compra` AS STRING)), '^0+', '')),
    'num_item_requisicao_compra', 'string', COUNT_IF(CAST(`num_item_requisicao_compra` AS STRING) IS NULL OR trim(CAST(`num_item_requisicao_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_requisicao_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_requisicao_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_requisicao_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_requisicao_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_requisicao_compra` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_empresa', 'string', COUNT_IF(CAST(`cod_empresa` AS STRING) IS NULL OR trim(CAST(`cod_empresa` AS STRING)) = ''), MIN(length(trim(CAST(`cod_empresa` AS STRING)))), MAX(length(trim(CAST(`cod_empresa` AS STRING)))), COUNT_IF(trim(CAST(`cod_empresa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_empresa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_empresa` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_solicitacao_cotacao_coletiva` AS STRING), ''), COALESCE(CAST(`num_cotacao` AS STRING), ''), COALESCE(CAST(`num_item_cotacao` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_cotacao` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`dt_criacao_cotacao` AS STRING), ''), COALESCE(CAST(`dt_prazo_apresentacao_cotacao` AS STRING), ''), COALESCE(CAST(`hora_fechamento_cotacao` AS STRING), ''), COALESCE(CAST(`st_resposta_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`st_cotacao` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`qt_rodadas_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_titulo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_fornecedores` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento_subsequente` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_solicitacao_cotacao_coletiva` AS STRING), ''), COALESCE(CAST(`num_cotacao` AS STRING), ''), COALESCE(CAST(`num_item_cotacao` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_cotacao` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`dt_criacao_cotacao` AS STRING), ''), COALESCE(CAST(`dt_prazo_apresentacao_cotacao` AS STRING), ''), COALESCE(CAST(`hora_fechamento_cotacao` AS STRING), ''), COALESCE(CAST(`st_resposta_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`st_cotacao` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`qt_rodadas_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_titulo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_fornecedores` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento_subsequente` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_solicitacao_cotacao_coletiva` AS STRING), ''), COALESCE(CAST(`num_cotacao` AS STRING), ''), COALESCE(CAST(`num_item_cotacao` AS STRING), ''), COALESCE(CAST(`num_requisicao_compra` AS STRING), ''), COALESCE(CAST(`num_item_requisicao_compra` AS STRING), ''), COALESCE(CAST(`vl_cotacao` AS STRING), ''), COALESCE(CAST(`cod_condicao_pagamento` AS STRING), ''), COALESCE(CAST(`dt_criacao_cotacao` AS STRING), ''), COALESCE(CAST(`dt_prazo_apresentacao_cotacao` AS STRING), ''), COALESCE(CAST(`hora_fechamento_cotacao` AS STRING), ''), COALESCE(CAST(`st_resposta_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_usuario_criacao` AS STRING), ''), COALESCE(CAST(`st_cotacao` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`desc_item` AS STRING), ''), COALESCE(CAST(`qt_rodadas_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_titulo` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_fornecedores` AS STRING), ''), COALESCE(CAST(`dt_criacao_documento_subsequente` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

## 15. Amostra de linhas completas

In [0]:
-- AMOSTRA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
ORDER BY `num_cotacao`, `num_item_cotacao`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)') AS `cod_empresa`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_empresa` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_cotacao
SELECT COALESCE(NULLIF(trim(CAST(`st_cotacao` AS STRING)), ''), '(vazio)') AS `st_cotacao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`st_cotacao` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR st_resposta_fornecedor
SELECT COALESCE(NULLIF(trim(CAST(`st_resposta_fornecedor` AS STRING)), ''), '(vazio)') AS `st_resposta_fornecedor`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`st_resposta_fornecedor` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_grupo_fornecedores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_fornecedores` AS STRING)), ''), '(vazio)') AS `cod_grupo_fornecedores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_fornecedores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

⚠️ **Esta tabela não possui nenhuma coluna de auditoria de carga** — não há `dateingest`,
`dh_carga` nem equivalente.

**Consequência:** é impossível saber a que momento o dado se refere. Use o `DESCRIBE HISTORY`
da seção 1 para identificar a última gravação, e registre isso como limitação no relatório.

In [0]:
-- 17. FRESHNESS
-- Sem coluna de auditoria. Unica fonte e o historico Delta.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq LIMIT 10;

## 18. Análises específicas — ZGLMM381

### 18.1 Estrutura da cotação: coletiva → cotação → item → fornecedor

A tabela não tem clustering declarado. Esta célula mapeia a hierarquia real
para descobrir o grão.

In [0]:
-- 18.1 HIERARQUIA DA COTACAO
SELECT COUNT(*)                                                 AS linhas,
       COUNT(DISTINCT num_solicitacao_cotacao_coletiva)          AS solicitacoes_coletivas,
       COUNT(DISTINCT num_cotacao)                               AS cotacoes,
       COUNT(DISTINCT CONCAT_WS('|', num_cotacao, num_item_cotacao)) AS itens_de_cotacao,
       COUNT(DISTINCT cod_fornecedor)                            AS fornecedores,
       COUNT(DISTINCT num_requisicao_compra)                     AS requisicoes_origem,
       COUNT(DISTINCT cod_material)                              AS materiais
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

In [0]:
-- 18.1b COTACOES POR SOLICITACAO COLETIVA
SELECT cotacoes_na_coletiva, COUNT(*) AS solicitacoes
FROM (SELECT num_solicitacao_cotacao_coletiva,
             COUNT(DISTINCT num_cotacao) AS cotacoes_na_coletiva
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
       WHERE trim(COALESCE(num_solicitacao_cotacao_coletiva,'')) <> ''
       GROUP BY num_solicitacao_cotacao_coletiva)
GROUP BY cotacoes_na_coletiva
ORDER BY cotacoes_na_coletiva
LIMIT 40;

In [0]:
-- 18.1c FORNECEDORES POR ITEM DE COTACAO
SELECT fornecedores_no_item, COUNT(*) AS itens
FROM (SELECT num_cotacao, num_item_cotacao,
             COUNT(DISTINCT cod_fornecedor) AS fornecedores_no_item
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq GROUP BY num_cotacao, num_item_cotacao)
GROUP BY fornecedores_no_item
ORDER BY fornecedores_no_item
LIMIT 40;

### 18.2 Rodadas de negociação

`qt_rodadas_fornecedor` é `bigint` e sugere contagem de rodadas.
Esta célula confirma o domínio real.

In [0]:
-- 18.2 RODADAS DE NEGOCIACAO
SELECT qt_rodadas_fornecedor,
       COUNT(*)                          AS linhas,
       COUNT(DISTINCT num_cotacao)       AS cotacoes,
       ROUND(AVG(vl_cotacao), 2)         AS valor_medio,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY qt_rodadas_fornecedor
ORDER BY qt_rodadas_fornecedor
LIMIT 40;

### 18.3 Status da cotação × resposta do fornecedor

Cruza os dois campos de status para achar combinações inconsistentes —
por exemplo, cotação fechada sem resposta registrada.

In [0]:
-- 18.3 STATUS x RESPOSTA
SELECT COALESCE(NULLIF(trim(st_cotacao), ''), '(vazio)')            AS st_cotacao,
       COALESCE(NULLIF(trim(st_resposta_fornecedor), ''), '(vazio)') AS st_resposta_fornecedor,
       COUNT(*)                                     AS linhas,
       COUNT_IF(vl_cotacao IS NOT NULL AND vl_cotacao <> 0) AS com_valor_cotado,
       ROUND(AVG(vl_cotacao), 2)                    AS valor_medio
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
GROUP BY COALESCE(NULLIF(trim(st_cotacao), ''), '(vazio)'),
         COALESCE(NULLIF(trim(st_resposta_fornecedor), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 30;

In [0]:
-- 18.3b COMBINACOES SUSPEITAS
SELECT 'com valor cotado MAS sem resposta do fornecedor' AS situacao,
       COUNT_IF(vl_cotacao IS NOT NULL AND vl_cotacao <> 0
            AND trim(COALESCE(st_resposta_fornecedor,'')) = '') AS linhas
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
UNION ALL
SELECT 'com documento subsequente MAS sem valor cotado',
       COUNT_IF(dt_criacao_documento_subsequente IS NOT NULL
            AND (vl_cotacao IS NULL OR vl_cotacao = 0))
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
UNION ALL
SELECT 'sem vinculo com requisicao de compra',
       COUNT_IF(trim(COALESCE(num_requisicao_compra,'')) = '')
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
UNION ALL
SELECT 'prazo de apresentacao anterior a criacao',
       COUNT_IF(dt_prazo_apresentacao_cotacao < dt_criacao_cotacao)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
ORDER BY linhas DESC;

### 18.4 Precisão de `vl_cotacao`

O campo é `decimal(38,6)` — precisão muito acima do usual para valor monetário.
Esta célula verifica se as casas decimais extras têm uso real.

In [0]:
-- 18.4 PRECISAO DE vl_cotacao
SELECT COUNT(*)                                                       AS linhas,
       COUNT_IF(vl_cotacao IS NULL)                                   AS nulos,
       COUNT_IF(vl_cotacao = 0)                                       AS zerados,
       COUNT_IF(vl_cotacao < 0)                                       AS negativos,
       COUNT_IF(vl_cotacao = ROUND(vl_cotacao, 2))                    AS cabem_em_2_casas,
       COUNT_IF(vl_cotacao <> ROUND(vl_cotacao, 2))                   AS usam_mais_de_2_casas,
       ROUND(MIN(vl_cotacao), 6)                                      AS minimo,
       ROUND(MAX(vl_cotacao), 6)                                      AS maximo,
       CASE WHEN COUNT_IF(vl_cotacao <> ROUND(vl_cotacao, 2)) = 0
            THEN 'decimal(38,6) e superdimensionado - 2 casas bastariam'
            ELSE 'as casas extras tem uso real' END                   AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

### 18.5 Ciclo temporal da cotação

Sequência esperada: criação → prazo de apresentação → documento subsequente.

In [0]:
-- 18.5 CICLO TEMPORAL
SELECT COUNT(*)                                                           AS linhas,
       COUNT_IF(dt_criacao_cotacao IS NULL)                               AS sem_dt_criacao,
       COUNT_IF(dt_prazo_apresentacao_cotacao IS NULL)                    AS sem_prazo,
       COUNT_IF(dt_criacao_documento_subsequente IS NULL)                 AS sem_doc_subsequente,
       COUNT_IF(dt_prazo_apresentacao_cotacao < dt_criacao_cotacao)       AS prazo_antes_da_criacao,
       COUNT_IF(dt_criacao_documento_subsequente < dt_criacao_cotacao)    AS doc_antes_da_criacao,
       ROUND(AVG(datediff(dt_prazo_apresentacao_cotacao, dt_criacao_cotacao)), 1) AS media_dias_ate_prazo,
       ROUND(AVG(datediff(dt_criacao_documento_subsequente, dt_criacao_cotacao)), 1) AS media_dias_ate_doc
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq
ORDER BY `num_cotacao`, `num_item_cotacao`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq),
g AS (
  SELECT 'num_cotacao' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_cotacao + num_item_cotacao' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_cotacao`, `num_item_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_cotacao + num_item_cotacao + cod_fornecedor' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_cotacao`, `num_item_cotacao`, `cod_fornecedor` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
UNION ALL
  SELECT 'num_solicitacao_cotacao_coletiva + num_cotacao + num_item_cotacao' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_solicitacao_cotacao_coletiva`, `num_cotacao`, `num_item_cotacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm381_evolucao_rfq)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM381' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'tbl_ds_pro_zglmm381_evolucao_rfq'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '23'
UNION ALL SELECT 'IDENTIFICACAO', 'clustering declarado',
       '(nenhum)'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a ZGLMM381 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] Alertas de partida verificados (seção 18)
